In [ ]:
!python -m pip install -q 'git+https://github.com/facebookresearch/detectron2.git'

!pip install -q pyyaml==5.1

In [ ]:
import torch, detectron2
print("PyTorch version:", torch.__version__)
print("Detectron2 version:", detectron2.__version__)
print("GPU is available:", torch.cuda.is_available())

In [ ]:
!pip install -q roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="dapCMkEAGqQDBdjxPMQO")
project = rf.workspace("yolo-plant-ditection").project("vr-tsd-rdkbq")
version = project.version(1)
dataset = version.download("coco")

In [ ]:
from detectron2.data import build_detection_train_loader
from detectron2.data import DatasetMapper
import detectron2.data.transforms as T
from detectron2.engine import DefaultTrainer

class CustomTrainer(DefaultTrainer):
    @classmethod
    def build_evaluator(cls, cfg, dataset_name, output_folder=None):
        if output_folder is None:
            output_folder = os.path.join(cfg.OUTPUT_DIR, "inference")
            os.makedirs(output_folder, exist_ok=True)
        return COCOEvaluator(dataset_name, output_dir=output_folder)

    @classmethod
    def build_train_loader(cls, cfg):
        mapper = DatasetMapper(
            cfg,
            is_train=True,
            augmentations=[
                T.RandomFlip(prob=0.5),
                T.RandomBrightness(0.8, 1.2),
                T.RandomContrast(0.8, 1.2),
                T.RandomSaturation(0.8, 1.2),
                T.ResizeShortestEdge(
                    short_edge_length=(512,544,576,608,640,672,704),
                    max_size=800,
                    sample_style="choice"
                )
            ]
        )

        return build_detection_train_loader(
            cfg,
            mapper=mapper
        )

    @classmethod
    def build_evaluator(cls, cfg, dataset_name, output_folder=None):
        return COCOEvaluator(dataset_name, output_dir=cfg.OUTPUT_DIR)

In [ ]:
from detectron2.data.datasets import register_coco_instances
from detectron2.config import get_cfg
from detectron2 import model_zoo
import os
from detectron2.evaluation import COCOEvaluator
import warnings
import logging

warnings.filterwarnings("ignore")

logging.getLogger("fvcore").setLevel(logging.ERROR)

logging.getLogger("pycocotools").setLevel(logging.ERROR)

register_coco_instances(
    "my_dataset_train",
    {},
    "/content/VR-TSD-1/train/_annotations.coco.json",
    "/content/VR-TSD-1/train"
)

register_coco_instances(
    "my_dataset_val",
    {},
    "/content/VR-TSD-1/test/_annotations.coco.json",
    "/content/VR-TSD-1/test"
)


cfg = get_cfg()

cfg.merge_from_file(
    model_zoo.get_config_file(
        "COCO-Detection/retinanet_R_50_FPN_3x.yaml"
    )
)

cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url("COCO-Detection/retinanet_R_50_FPN_3x.yaml") # Or faster_rcnn and another, check name of yaml in detectron git

cfg.DATASETS.TRAIN = ("my_dataset_train",)
cfg.DATASETS.TEST = ("my_dataset_val",)

cfg.MODEL.RETINANET.NUM_CLASSES = 58

cfg.DATALOADER.NUM_WORKERS = 8

cfg.INPUT.MIN_SIZE_TRAIN = (640,)
cfg.INPUT.MAX_SIZE_TRAIN = 800

cfg.INPUT.MIN_SIZE_TEST = 640
cfg.INPUT.MAX_SIZE_TEST = 800

# 1. Thông số Solver
cfg.SOLVER.IMS_PER_BATCH = 64
cfg.SOLVER.BASE_LR = 0.04

cfg.SOLVER.WARMUP_ITERS = 1000
cfg.SOLVER.WARMUP_FACTOR = 0.0001


cfg.SOLVER.MAX_ITER = 15010
cfg.SOLVER.STEPS = (6230, 8010)
cfg.SOLVER.GAMMA = 0.1

cfg.SOLVER.OPTIMIZER = "SGD"
cfg.SOLVER.MOMENTUM = 0.9
cfg.SOLVER.AMP.ENABLED = True

# 3. Thông số Model
cfg.MODEL.RETINANET.NUM_CLASSES = 58

cfg.OUTPUT_DIR = "/content/drive/MyDrive/RetinaNet"
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

cfg.TEST.EVAL_PERIOD = 1000

trainer = CustomTrainer(cfg)
trainer.resume_or_load(resume=True)
trainer.train()

In [ ]:
import os
from detectron2.evaluation import COCOEvaluator, inference_on_dataset
from detectron2.data import build_detection_test_loader
from detectron2.engine import DefaultPredictor

register_coco_instances(
    "test",
    {},
    "/content/VR-TSD-1/test/_annotations.coco.json",
    "/content/VR-TSD-1/test"
)

cfg.MODEL.WEIGHTS = os.path.join(cfg.OUTPUT_DIR, "") #Your best pth model

cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.5

predictor = DefaultPredictor(cfg)

val_dataset_name = "test"

evaluator = COCOEvaluator(val_dataset_name, output_dir=cfg.OUTPUT_DIR)

val_loader = build_detection_test_loader(cfg, val_dataset_name)

evaluation_results = inference_on_dataset(predictor.model, val_loader, evaluator)

print("\n" + "="*50)
print("EVALUATION RESULT:")
print("="*50)
print(evaluation_results)